# Phase 6 quick baseline-only candidates

Upload only the original `submission.zip` that scored **0.811055**. This CPU notebook removes repeated sentences and makes two length variants. It creates valid submission ZIPs and downloads the most conservative changed version. **No GPU, model weights, dataset, Hugging Face token, or Drive is needed.** No score increase is guaranteed; keep the scored baseline unless Codabench proves otherwise.


In [ ]:
from google.colab import files
from pathlib import Path
import io, json, re, zipfile, statistics

print("Upload ONLY the original Phase 6 submission.zip that scored 0.811055")
uploaded = files.upload()
assert len(uploaded) == 1, "Select exactly one ZIP"
payload = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert archive.namelist() == ["prediction.json"], "ZIP must contain prediction.json at root"
    baseline = json.loads(archive.read("prediction.json"))
assert isinstance(baseline, list) and len(baseline) == 100
assert len({str(row["id"]) for row in baseline}) == 100
assert all(isinstance(row["summary"], str) and row["summary"].strip() for row in baseline)
print("Loaded 100 predictions; average words:",
      round(statistics.mean(len(row["summary"].split()) for row in baseline), 1))
DEST = Path("/content/phase6_baseline_only_candidates")
DEST.mkdir(parents=True, exist_ok=True)


In [ ]:
def tokens(text):
    return set(re.findall(r"[a-zA-Z]{3,}|[0-9]+", text.lower()))

def dedup(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    kept, seen = [], []
    removed = 0
    for part in parts:
        words = tokens(part)
        if len(words) < 5:
            kept.append(part)
            continue
        repeated = any(len(words & prior) / max(1, len(words | prior)) >= 0.82
                       for prior in seen)
        if not repeated:
            kept.append(part)
            seen.append(words)
        else:
            removed += 1
    return (" ".join(kept).strip() if removed else text.strip()) or text.strip()

def trim(text, max_words):
    if len(text.split()) <= max_words:
        return text.strip()
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    chosen, length = [], 0
    for part in parts:
        count = len(part.split())
        if length + count > max_words:
            break
        chosen.append(part)
        length += count
    if length >= max_words * 0.65:
        return " ".join(chosen).strip()
    return " ".join(text.split()[:max_words]).strip()

def write_candidate(name, transform):
    rows = [{"id": str(row["id"]), "summary": transform(row["summary"])}
            for row in baseline]
    changed = sum(a["summary"] != b["summary"] for a, b in zip(rows, baseline))
    assert len(rows) == 100 and all(row["summary"] for row in rows)
    if changed == 0:
        print(name, "made no changes; skipped")
        return None
    target = DEST / (name + ".zip")
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("prediction.json", json.dumps(rows, ensure_ascii=False) + "\n")
    with zipfile.ZipFile(target) as archive:
        assert archive.namelist() == ["prediction.json"] and archive.testzip() is None
    print("READY:", target, "changed", changed, "of 100")
    return target

dedup_path = write_candidate("baseline_deduplicated", dedup)
trim230_path = write_candidate("baseline_trim230", lambda s: trim(s, 230))
trim200_path = write_candidate("baseline_trim200", lambda s: trim(s, 200))
combo_path = write_candidate("baseline_dedup_trim230", lambda s: trim(dedup(s), 230))

primary = dedup_path or trim230_path or trim200_path
if primary:
    print("Downloading first candidate:", primary.name)
    files.download(str(primary))
else:
    print("No candidate changed the baseline. Keep the scored 0.811055 ZIP.")
print("Other candidate ZIPs, if created, are under", DEST)
print("Keep the original 0.811055 submission unless a candidate scores higher.")
